# EXP-005: Cross-Model Comparison

Compare embedding models for semantic navigation in autonomous driving scenes.

**Models:**
- `eva02_e`: EVA02-E-14-plus (4.4B) — MIM + size
- `openclip_bigg`: ViT-bigG-14 (2.5B) — Pure CLIP
- `siglip2_so400m`: SigLIP2 SO400M (400M) — Sigmoid loss
- `openai_clip_l`: ViT-L-14-336 (428M) — Baseline
- `eva02_l`: EVA02-L-14-336 (428M) — MIM comparison
- `finetuned`: EVA02-E finetuned on driving scenes

In [25]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.precision', 3)

## 1. Load Results

In [26]:
DATA_DIR = Path("../../data/EXP-005")

# Configuration
RUN_TAG = "v2"  # Which run to analyze
KEY_SET = "top"  # "all" or "top"

RUN_DIR = DATA_DIR / RUN_TAG

# Model metadata
MODEL_INFO = {
    "eva02_e": {"params": "4.4B", "type": "MIM+CLIP", "color": "#636EFA"},
    "openclip_bigg": {"params": "2.5B", "type": "CLIP", "color": "#EF553B"},
    "siglip2_so400m": {"params": "400M", "type": "SigLIP", "color": "#00CC96"},
    "openai_clip_l": {"params": "428M", "type": "CLIP", "color": "#AB63FA"},
    "eva02_l": {"params": "428M", "type": "MIM+CLIP", "color": "#FFA15A"},
    "finetuned": {"params": "4.4B*", "type": "Finetuned", "color": "#19D3F3"},
}

def find_run(model_name: str, key_set: str) -> Path | None:
    """Find latest run for model and key set."""
    pattern = f"{model_name}_{key_set}_*"
    runs = list(RUN_DIR.glob(pattern))
    if not runs:
        return None
    return max(runs, key=lambda p: p.name)

# Load all results
results = {}
for name in MODEL_INFO.keys():
    run_dir = find_run(name, KEY_SET)
    if run_dir:
        with open(run_dir / "analysis.json") as f:
            results[name] = json.load(f)
        print(f"✓ {name}: {run_dir.name}")
    else:
        print(f"✗ {name}: not found")

print(f"\nLoaded {len(results)} models from {RUN_DIR}")
print(f"Key set: {KEY_SET}")
if results:
    sample = next(iter(results.values()))
    print(f"N scenes: {sample.get('n_scenes', '?')} ({sample.get('n_anchors', '?')} anchors)")

✓ eva02_e: eva02_e_top_20260129_043407
✓ openclip_bigg: openclip_bigg_top_20260129_043407
✓ siglip2_so400m: siglip2_so400m_top_20260129_043407
✓ openai_clip_l: openai_clip_l_top_20260129_043407
✓ eva02_l: eva02_l_top_20260129_043407
✓ finetuned: finetuned_top_20260129_043407

Loaded 6 models from ../../data/EXP-005/v2
Key set: top
N scenes: 2600 (100 anchors)


## 2. Text Alignment Accuracy (Primary Metric)

In [27]:
# Extract text alignment metrics
text_align_data = []
for name, res in results.items():
    ta = res.get("text_alignment", {})
    for key, metrics in ta.items():
        text_align_data.append({
            "model": name,
            "key": key,
            "accuracy": metrics.get("accuracy", 0),
            "mean_rank": metrics.get("mean_rank", 0),
            "n_samples": metrics.get("n_samples", 0),
        })

df_text = pd.DataFrame(text_align_data)
df_text_pivot = df_text.pivot(index="key", columns="model", values="accuracy")

# Reorder columns by mean accuracy
model_order = df_text.groupby("model")["accuracy"].mean().sort_values(ascending=False).index.tolist()
df_text_pivot = df_text_pivot[model_order]

# Add mean row
df_text_pivot.loc["MEAN"] = df_text_pivot.mean()

print("Text Alignment Accuracy by Key and Model:")
df_text_pivot.style.format("{:.1%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1)

Text Alignment Accuracy by Key and Model:


model,openclip_bigg,eva02_e,openai_clip_l,eva02_l,siglip2_so400m,finetuned
key,,,,,,
depth_complexity,70.0%,76.0%,48.0%,45.0%,14.0%,58.0%
occlusion_level,37.0%,31.0%,35.0%,29.0%,37.0%,18.0%
required_action,15.0%,15.0%,27.0%,16.0%,15.0%,17.0%
road_type,28.0%,29.0%,28.0%,25.0%,18.0%,2.0%
time_of_day,91.0%,91.0%,66.0%,91.0%,77.0%,29.0%
weather,86.0%,82.0%,77.0%,72.0%,80.0%,30.0%
MEAN,54.5%,54.0%,46.8%,46.3%,40.2%,25.7%


In [28]:
# Heatmap visualization
df_plot = df_text_pivot.drop("MEAN")

fig = px.imshow(
    df_plot.values,
    x=df_plot.columns,
    y=df_plot.index,
    color_continuous_scale="RdYlGn",
    aspect="auto",
    text_auto=".0%",
    title=f"Text Alignment Accuracy (key_set={KEY_SET})",
    labels={"color": "Accuracy"},
)
fig.update_layout(height=400)
fig.show()

In [29]:
# Bar chart: Mean accuracy per model
mean_acc = df_text.groupby("model")["accuracy"].mean().sort_values(ascending=True)
colors = [MODEL_INFO.get(m, {}).get("color", "gray") for m in mean_acc.index]

fig = go.Figure(go.Bar(
    x=mean_acc.values,
    y=mean_acc.index,
    orientation="h",
    marker_color=colors,
    text=[f"{v:.1%}" for v in mean_acc.values],
    textposition="outside",
))
fig.update_layout(
    title="Mean Text Alignment Accuracy",
    xaxis_title="Accuracy",
    xaxis_tickformat=".0%",
    height=350,
    xaxis_range=[0, 0.7],
)
fig.show()

## 3. Finetuned Model: Direct Prediction vs Text Alignment

In [30]:
# Compare finetuned model's text alignment vs direct prediction
if "finetuned" in results:
    ft = results["finetuned"]
    ta = ft.get("text_alignment", {})
    pm = ft.get("prediction_metrics", {})
    
    comparison_data = []
    for key in set(ta.keys()) | set(pm.keys()):
        comparison_data.append({
            "key": key,
            "text_alignment": ta.get(key, {}).get("accuracy", float("nan")),
            "direct_prediction": pm.get(key, {}).get("accuracy", float("nan")),
        })
    
    df_ft = pd.DataFrame(comparison_data).set_index("key").sort_values("direct_prediction", ascending=False)
    
    print("Finetuned Model: Text Alignment vs Direct Prediction")
    display(df_ft.style.format("{:.1%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))
    
    # Bar chart comparison
    df_melt = df_ft.reset_index().melt(id_vars="key", var_name="method", value_name="accuracy")
    
    fig = px.bar(
        df_melt,
        x="key",
        y="accuracy",
        color="method",
        barmode="group",
        title="Finetuned Model: Text Alignment vs Direct Prediction",
        color_discrete_map={"text_alignment": "#EF553B", "direct_prediction": "#00CC96"},
    )
    fig.update_layout(yaxis_tickformat=".0%", height=400)
    fig.show()
else:
    print("No finetuned model results found.")

Finetuned Model: Text Alignment vs Direct Prediction


,text_alignment,direct_prediction
key,,
time_of_day,29.0%,88.0%
pedestrians_present,nan%,85.0%
weather,30.0%,81.0%
depth_complexity,58.0%,79.0%
traffic_signals_visible,nan%,71.0%
required_action,17.0%,59.0%
occlusion_level,18.0%,58.0%
road_type,2.0%,nan%


## 4. Cluster Quality

In [31]:
# Extract cluster metrics
cluster_data = []
for name, res in results.items():
    cm = res.get("cluster_metrics", {})
    cluster_data.append({
        "model": name,
        "params": MODEL_INFO.get(name, {}).get("params", "?"),
        "type": MODEL_INFO.get(name, {}).get("type", "?"),
        "n_clusters": cm.get("n_clusters", 0),
        "noise_ratio": cm.get("noise_ratio", 1.0),
        "silhouette": cm.get("silhouette_score", float("nan")),
    })

df_cluster = pd.DataFrame(cluster_data).sort_values("noise_ratio")
df_cluster

,model,params,type,n_clusters,noise_ratio,silhouette
0,eva02_e,4.4B,MIM+CLIP,2,0.092,0.095
3,openai_clip_l,428M,CLIP,2,0.229,0.037
1,openclip_bigg,2.5B,CLIP,2,0.246,0.121
4,eva02_l,428M,MIM+CLIP,8,0.407,-0.050
2,siglip2_so400m,400M,SigLIP,2,0.409,0.028
5,finetuned,4.4B*,Finetuned,2,0.576,0.194


In [32]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Noise Ratio (lower=better)", "Silhouette Score (higher=better)"])

colors = [MODEL_INFO.get(m, {}).get("color", "gray") for m in df_cluster["model"]]

fig.add_trace(
    go.Bar(x=df_cluster["model"], y=df_cluster["noise_ratio"], marker_color=colors, showlegend=False),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=df_cluster["model"], y=df_cluster["silhouette"], marker_color=colors, showlegend=False),
    row=1, col=2
)

fig.update_layout(title="HDBSCAN Cluster Quality", height=400)
fig.update_yaxes(tickformat=".0%", row=1, col=1)
fig.show()

## 5. Size vs Performance

In [33]:
# Combine metrics for size comparison
param_map = {"400M": 0.4, "428M": 0.428, "2.5B": 2.5, "4.4B": 4.4, "4.4B*": 4.4}

size_data = []
for name, res in results.items():
    cm = res.get("cluster_metrics", {})
    ta = res.get("text_alignment", {})
    
    mean_text_acc = np.nanmean([m.get("accuracy", np.nan) for m in ta.values()]) if ta else np.nan
    
    size_data.append({
        "model": name,
        "params_B": param_map.get(MODEL_INFO.get(name, {}).get("params", "0"), 0),
        "type": MODEL_INFO.get(name, {}).get("type", "?"),
        "noise_ratio": cm.get("noise_ratio", 1),
        "mean_text_acc": mean_text_acc,
    })

df_size = pd.DataFrame(size_data)

fig = px.scatter(
    df_size,
    x="params_B",
    y="mean_text_acc",
    color="type",
    size=[max(0.3, p) for p in df_size["params_B"]],
    text="model",
    title="Model Size vs Text Alignment Accuracy",
    labels={"params_B": "Parameters (Billions)", "mean_text_acc": "Mean Text Alignment Accuracy"},
)
fig.update_traces(textposition="top center")
fig.update_layout(height=450, yaxis_tickformat=".0%")
fig.show()

## 6. Summary Rankings

In [34]:
# Exclude finetuned from rankings (different evaluation paradigm)
pretrained_models = [m for m in results.keys() if m != "finetuned"]

print("=" * 50)
print("SUMMARY RANKINGS (Pretrained Models Only)")
print("=" * 50)

# 1. Text alignment
print("\n1. Text Alignment Accuracy (higher = better):")
text_rank = df_text[df_text["model"].isin(pretrained_models)].groupby("model")["accuracy"].mean().sort_values(ascending=False)
for i, (model, acc) in enumerate(text_rank.items(), 1):
    params = MODEL_INFO.get(model, {}).get("params", "?")
    print(f"   {i}. {model} ({params}): {acc:.1%}")

# 2. Noise ratio
print("\n2. Noise Ratio (lower = better clustering):")
noise_rank = df_cluster[df_cluster["model"].isin(pretrained_models)].sort_values("noise_ratio")
for i, (_, row) in enumerate(noise_rank.iterrows(), 1):
    print(f"   {i}. {row['model']}: {row['noise_ratio']:.1%}")

# 3. Best model per key
print("\n3. Best Model per Key:")
df_pretrained = df_text[df_text["model"].isin(pretrained_models)]
best_per_key = df_pretrained.loc[df_pretrained.groupby("key")["accuracy"].idxmax()]
for _, row in best_per_key.iterrows():
    print(f"   {row['key']}: {row['model']} ({row['accuracy']:.1%})")

SUMMARY RANKINGS (Pretrained Models Only)

1. Text Alignment Accuracy (higher = better):
   1. openclip_bigg (2.5B): 54.5%
   2. eva02_e (4.4B): 54.0%
   3. openai_clip_l (428M): 46.8%
   4. eva02_l (428M): 46.3%
   5. siglip2_so400m (400M): 40.2%

2. Noise Ratio (lower = better clustering):
   1. eva02_e: 9.2%
   2. openai_clip_l: 22.9%
   3. openclip_bigg: 24.6%
   4. eva02_l: 40.7%
   5. siglip2_so400m: 40.9%

3. Best Model per Key:
   depth_complexity: eva02_e (76.0%)
   occlusion_level: openclip_bigg (37.0%)
   required_action: openai_clip_l (27.0%)
   road_type: eva02_e (29.0%)
   time_of_day: eva02_e (91.0%)
   weather: openclip_bigg (86.0%)


In [35]:
# Winner analysis
print("\n" + "=" * 50)
print("KEY FINDINGS")
print("=" * 50)

winner = text_rank.idxmax()
winner_acc = text_rank.max()
runner_up = text_rank.index[1]
runner_up_acc = text_rank.iloc[1]

print(f"\n• Best text alignment: {winner} ({winner_acc:.1%})")
print(f"  Runner-up: {runner_up} ({runner_up_acc:.1%})")
print(f"  Gap: {(winner_acc - runner_up_acc)*100:.1f} percentage points")

# Size efficiency
best_small = df_size[(df_size["params_B"] < 1) & (df_size["model"].isin(pretrained_models))].sort_values("mean_text_acc", ascending=False).iloc[0]
print(f"\n• Best <1B model: {best_small['model']} ({best_small['mean_text_acc']:.1%})")

# MIM comparison
eva_l = df_text[df_text["model"] == "eva02_l"]["accuracy"].mean()
openai = df_text[df_text["model"] == "openai_clip_l"]["accuracy"].mean()
print(f"\n• MIM effect (same size): eva02_l={eva_l:.1%} vs openai_clip_l={openai:.1%}")
print(f"  MIM {'helps' if eva_l > openai else 'hurts'}: {abs(eva_l-openai)*100:.1f}pp difference")

if "finetuned" in results:
    ft_pred = np.mean([m["accuracy"] for m in results["finetuned"].get("prediction_metrics", {}).values()])
    ft_text = df_text[df_text["model"] == "finetuned"]["accuracy"].mean()
    print(f"\n• Finetuned model: text={ft_text:.1%}, prediction={ft_pred:.1%}")
    print(f"  Finetuning breaks text alignment but enables {ft_pred:.1%} direct prediction")


KEY FINDINGS

• Best text alignment: openclip_bigg (54.5%)
  Runner-up: eva02_e (54.0%)
  Gap: 0.5 percentage points

• Best <1B model: openai_clip_l (46.8%)

• MIM effect (same size): eva02_l=46.3% vs openai_clip_l=46.8%
  MIM hurts: 0.5pp difference

• Finetuned model: text=25.7%, prediction=74.4%
  Finetuning breaks text alignment but enables 74.4% direct prediction


In [36]:
# Save summary
summary = {
    "run_tag": RUN_TAG,
    "key_set": KEY_SET,
    "n_scenes": results[list(results.keys())[0]].get("n_scenes", 0),
    "n_anchors": results[list(results.keys())[0]].get("n_anchors", 0),
    "models": list(results.keys()),
    "rankings": {
        "text_alignment": text_rank.to_dict(),
        "noise_ratio": df_cluster.set_index("model")["noise_ratio"].to_dict(),
    },
    "best_per_key": best_per_key.set_index("key")[["model", "accuracy"]].to_dict(orient="index"),
    "winner": winner,
}

output_path = RUN_DIR / f"comparison_analysis_{KEY_SET}.json"
with open(output_path, "w") as f:
    json.dump(summary, f, indent=2)
    
print(f"\nSummary saved to {output_path}")


Summary saved to ../../data/EXP-005/v2/comparison_analysis_top.json


## 7. Latent Space Visualization (3D UMAP)

Visualize how each model structures the embedding space, colored by semantic anchor labels.

In [37]:
# Load UMAP coordinates and classifications for each model
import pickle

umap_data = {}

for name in MODEL_INFO.keys():
    run_dir = find_run(name, KEY_SET)
    if not run_dir:
        continue
    
    # Load UMAP coordinates from umap_coords.npz
    umap_path = run_dir / "umap_coords.npz"
    if umap_path.exists():
        data = np.load(umap_path, allow_pickle=True)
        umap_data[name] = {
            "coords_3d": data["coords"],
            "scene_ids": data["scene_ids"].tolist(),
        }
        print(f"✓ {name}: {len(umap_data[name]['scene_ids'])} scenes")
    else:
        print(f"✗ {name}: umap_coords.npz not found")

# Load anchor classifications from EXP-001 and convert to dict format
exp001_path = Path("../../data/EXP-001/scene_classifications.json")
anchor_classifications = {}

if exp001_path.exists():
    with open(exp001_path) as f:
        raw_data = json.load(f)
    
    # Convert list format to dict keyed by clip_id, extracting nested values
    for item in raw_data.get("classifications", []):
        clip_id = item["clip_id"]
        cls = item["classification"]
        
        # Extract actual values from nested dicts
        flat_cls = {}
        for key, val in cls.items():
            if isinstance(val, dict) and key in val:
                # Nested format: {"weather": {"reasoning": ..., "weather": "foggy"}}
                flat_cls[key] = val[key]
            elif isinstance(val, dict) and "value" in val:
                flat_cls[key] = val["value"]
            else:
                flat_cls[key] = val
        
        anchor_classifications[clip_id] = flat_cls
    
    print(f"\nLoaded {len(anchor_classifications)} anchor classifications from EXP-001")
    
    # Show sample
    sample_id = list(anchor_classifications.keys())[0]
    sample = anchor_classifications[sample_id]
    print(f"Sample: weather={sample.get('weather')}, time_of_day={sample.get('time_of_day')}")
else:
    print("⚠ No anchor classifications found")

✓ eva02_e: 2600 scenes
✓ openclip_bigg: 2600 scenes
✓ siglip2_so400m: 2600 scenes
✓ openai_clip_l: 2600 scenes
✓ eva02_l: 2600 scenes
✓ finetuned: 2600 scenes

Loaded 100 anchor classifications from EXP-001
Sample: weather=foggy, time_of_day=night


In [38]:
# Helper: Create 3D scatter colored by a semantic key
def plot_latent_space(model_name: str, color_by: str, show_anchors_only: bool = True):
    """
    Plot 3D UMAP coordinates colored by semantic key.
    
    Args:
        model_name: Which model's embeddings to plot
        color_by: Semantic key name (e.g., "weather", "time_of_day")
        show_anchors_only: If True, only show the 100 labeled anchor scenes
    """
    if model_name not in umap_data:
        print(f"No UMAP data for {model_name}")
        return None
    
    data = umap_data[model_name]
    coords = data["coords_3d"]
    scene_ids = data["scene_ids"]
    
    if coords is None:
        print(f"No UMAP coordinates for {model_name}")
        return None
    
    # Filter to anchors only if requested (default: True for labeled visualization)
    if show_anchors_only:
        anchor_mask = [sid in anchor_classifications for sid in scene_ids]
        coords = coords[anchor_mask]
        scene_ids = [sid for sid, m in zip(scene_ids, anchor_mask) if m]
        if len(scene_ids) == 0:
            print(f"No anchor scenes found for {model_name}")
            return None
    
    # Get color values from classifications
    color_values = []
    for sid in scene_ids:
        cls = anchor_classifications.get(sid, {})
        val = cls.get(color_by, "unknown")
        # Handle nested dicts (from EXP-001 format)
        if isinstance(val, dict):
            val = val.get("value", str(val))
        color_values.append(str(val))
    
    # Build hover text
    hover_texts = []
    for sid in scene_ids:
        cls = anchor_classifications.get(sid, {})
        parts = [f"<b>{sid[:16]}...</b>"]
        for key in ["weather", "time_of_day", "road_type", "occlusion_level", "required_action"]:
            val = cls.get(key, "?")
            if isinstance(val, dict):
                val = val.get("value", "?")
            parts.append(f"{key}: {val}")
        hover_texts.append("<br>".join(parts))
    
    # Create figure
    fig = px.scatter_3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        color=color_values,
        hover_name=hover_texts,
        title=f"{model_name} — Color: {color_by}" + (f" (N={len(scene_ids)} anchors)" if show_anchors_only else ""),
        labels={"x": "UMAP 1", "y": "UMAP 2", "z": "UMAP 3", "color": color_by},
    )
    fig.update_traces(marker=dict(size=5, opacity=0.8))
    fig.update_layout(height=500, margin=dict(l=0, r=0, t=50, b=0))
    
    return fig

### 7.1 Best Models: Latent Space by Semantic Key

Compare how the top 2 models (openclip_bigg and eva02_e) structure the space for key semantic attributes.

In [39]:
# Visualize top model (openclip_bigg) colored by different semantic keys
TOP_MODEL = "openclip_bigg"
SEMANTIC_KEYS = ["weather", "time_of_day", "depth_complexity", "occlusion_level"]

for key in SEMANTIC_KEYS:
    fig = plot_latent_space(TOP_MODEL, color_by=key, show_anchors_only=True)
    if fig:
        fig.show()

### 7.2 Model Comparison: Same Key Across Models

Compare how different models structure the space for the same semantic key (time_of_day — highest alignment).

In [40]:
# Compare all models for time_of_day (best-aligned key)
COMPARE_KEY = "time_of_day"
MODELS_TO_COMPARE = ["openclip_bigg", "eva02_e", "openai_clip_l", "siglip2_so400m"]

for model in MODELS_TO_COMPARE:
    fig = plot_latent_space(model, color_by=COMPARE_KEY, show_anchors_only=True)
    if fig:
        fig.show()

### 7.3 Full Dataset View: Anchors in Context

Visualize all 2,600 scenes with labeled anchors highlighted (anchors in color, superset in gray).

In [41]:
# Show full dataset with anchors highlighted
def plot_anchors_in_context(model_name: str, color_by: str = "weather"):
    """Plot all scenes with anchors colored by semantic key, superset in gray."""
    if model_name not in umap_data:
        print(f"No UMAP data for {model_name}")
        return None
    
    data = umap_data[model_name]
    coords = data["coords_3d"]
    scene_ids = data["scene_ids"]
    
    # Separate anchors and superset
    anchor_mask = np.array([sid in anchor_classifications for sid in scene_ids])
    
    fig = go.Figure()
    
    # Superset (gray, small, low opacity)
    superset_coords = coords[~anchor_mask]
    fig.add_trace(go.Scatter3d(
        x=superset_coords[:, 0],
        y=superset_coords[:, 1],
        z=superset_coords[:, 2],
        mode="markers",
        marker=dict(size=2, color="lightgray", opacity=0.3),
        name=f"Superset (N={len(superset_coords)})",
        hoverinfo="skip",
    ))
    
    # Anchors (colored by key)
    anchor_coords = coords[anchor_mask]
    anchor_ids = [sid for sid, m in zip(scene_ids, anchor_mask) if m]
    
    color_values = []
    hover_texts = []
    for sid in anchor_ids:
        cls = anchor_classifications.get(sid, {})
        val = cls.get(color_by, "unknown")
        if isinstance(val, dict):
            val = val.get("value", str(val))
        color_values.append(str(val))
        
        parts = [f"<b>{sid[:16]}...</b>"]
        for key in ["weather", "time_of_day", "road_type"]:
            v = cls.get(key, "?")
            if isinstance(v, dict):
                v = v.get("value", "?")
            parts.append(f"{key}: {v}")
        hover_texts.append("<br>".join(parts))
    
    # Get unique values for color mapping
    unique_vals = sorted(set(color_values))
    color_map = {v: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)] 
                 for i, v in enumerate(unique_vals)}
    
    for val in unique_vals:
        mask = [v == val for v in color_values]
        val_coords = anchor_coords[[i for i, m in enumerate(mask) if m]]
        val_hovers = [h for h, m in zip(hover_texts, mask) if m]
        
        fig.add_trace(go.Scatter3d(
            x=val_coords[:, 0],
            y=val_coords[:, 1],
            z=val_coords[:, 2],
            mode="markers",
            marker=dict(size=6, color=color_map[val], opacity=0.9),
            name=f"{color_by}={val}",
            text=val_hovers,
            hoverinfo="text",
        ))
    
    fig.update_layout(
        title=f"{model_name} — Anchors by {color_by} (in context of {len(superset_coords)} superset scenes)",
        scene=dict(xaxis_title="UMAP 1", yaxis_title="UMAP 2", zaxis_title="UMAP 3"),
        height=550,
        margin=dict(l=0, r=0, t=50, b=0),
    )
    
    return fig

# Show for top 2 models
for model in ["openclip_bigg", "eva02_e"]:
    fig = plot_anchors_in_context(model, color_by="time_of_day")
    if fig:
        fig.show()

### 7.4 Full Interactive Visualizations

For deeper exploration (toggle edges, text anchors, different coloring), open the HTML files:

In [42]:
# List available visualization HTML files
print("Full interactive visualizations (open in browser):\n")
for name in MODEL_INFO.keys():
    run_dir = find_run(name, KEY_SET)
    if run_dir:
        viz_path = run_dir / "visualization.html"
        if viz_path.exists():
            print(f"  {name}: {viz_path}")
        else:
            print(f"  {name}: (no visualization.html)")

Full interactive visualizations (open in browser):

  eva02_e: ../../data/EXP-005/v2/eva02_e_top_20260129_043407/visualization.html
  openclip_bigg: ../../data/EXP-005/v2/openclip_bigg_top_20260129_043407/visualization.html
  siglip2_so400m: ../../data/EXP-005/v2/siglip2_so400m_top_20260129_043407/visualization.html
  openai_clip_l: ../../data/EXP-005/v2/openai_clip_l_top_20260129_043407/visualization.html
  eva02_l: ../../data/EXP-005/v2/eva02_l_top_20260129_043407/visualization.html
  finetuned: ../../data/EXP-005/v2/finetuned_top_20260129_043407/visualization.html
